# Toy generation with ROOT output

This notebook shows how to generate unweighted pseudo-data and save it directly to ROOT with `uproot`. The default toy sampler is the Laura++-style accept-reject generator.

For the CP example, both charges are stored in a **single `DecayTree`** with the convention `charge=+1` for $B^+$ and `charge=-1` for $B^-$.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import uproot

from dalitzplotfitter import (
    BaBarFlatte,
    CPRealImag,
    DecayChannel,
    DecayModel,
    LASS,
    NonResonant,
    Parameter,
    Resonance,
    enable_x64,
    generate_cp_toy,
    generate_toy,
    plot_dalitz,
)

enable_x64()
output_dir = Path("toy_root_output")
output_dir.mkdir(exist_ok=True)


## Full $B^\pm\to K^\pm\pi^+\pi^-$ benchmark model

The ROOT example uses the same paper-inspired benchmark model as the main $B\to K\pi\pi$ notebooks:

$$K^*(892)^0 + (K\pi)_S^{\mathrm{LASS}} + \rho(770)^0 + f_0(980)_{\mathrm{Flatt\acute e}} + NR.$$


In [ ]:
truth_spec = {
    "Kstar892": (1.00, 0.00, +0.04, -0.03),
    "KpiS":     (1.40, -0.60, -0.10, +0.08),
    "rho770":   (0.65, 0.10, +0.06, +0.04),
    "f0_980":   (-0.20, 1.00, -0.05, +0.07),
    "NR":       (-0.50, 0.10, 0.00, 0.00),
}

truth = {}
shared = {}
for name, (x, y, dx, dy) in truth_spec.items():
    pars = (
        Parameter.coefficient(f"{name}.x", x, owner=name, fixed=(name == "Kstar892"), step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, fixed=(name == "Kstar892"), step=0.01),
        Parameter.coefficient(f"{name}.dx", dx, owner=name, fixed=(name == "NR"), step=0.01),
        Parameter.coefficient(f"{name}.dy", dy, owner=name, fixed=(name == "NR"), step=0.01),
    )
    shared[name] = CPRealImag(*pars)
    truth.update({p.name: p.value for p in pars})

def components(charge):
    c = {name: coeff.for_charge(charge) for name, coeff in shared.items()}
    return [
        Resonance(
            "Kstar892", (0, 2), c["Kstar892"],
            mass=0.8958, width=0.0474, spin=1,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "KpiS", (0, 2), c["KpiS"],
            lineshape=LASS(2.07, 3.32, 1.8),
            mass=1.425, width=0.270, spin=0,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "rho770", (1, 2), c["rho770"],
            mass=0.7753, width=0.1491, spin=1,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "f0_980", (1, 2), c["f0_980"],
            lineshape=BaBarFlatte(),
            mass=0.965, width=0.0, spin=0,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        NonResonant(c["NR"]),
    ]

plus_model = DecayModel(
    DecayChannel("B+", ("K+", "pi+", "pi-")),
    components(+1),
    normalization_method="square-dalitz",
    normalization_resolution=250,
    normalization_pair=(0, 2),
)
minus_model = DecayModel(
    DecayChannel("B-", ("K-", "pi-", "pi+")),
    components(-1),
    normalization_method="square-dalitz",
    normalization_resolution=250,
    normalization_pair=(0, 2),
)


## Non-CP toy saved directly to ROOT

`generate_toy` still returns the in-memory `PhaseSpaceSample`; `output_root=` only adds persistence. The default output tree is `DecayTree`.


In [ ]:
single_root = output_dir / "bplus_toy.root"

toy = generate_toy(
    plus_model,
    10_000,
    parameters=truth,
    seed=1901,
    output_root=single_root,
)

print("generated events:", toy.size)
print("ROOT file:", single_root)


In [ ]:
with uproot.open(single_root) as root_file:
    tree = root_file["DecayTree"]
    print("ROOT object type:", type(tree).__name__)
    print("entries:", tree.num_entries)
    print("branches:", tree.keys())

    arrays = tree.arrays(["s12", "s13", "s23", "weight"], library="np")

print("first s13 values:", arrays["s13"][:5])
print("unique toy weights:", np.unique(arrays["weight"]))


In [ ]:
plot_dalitz(toy, x="s13", y="s23", bins=80, title=r"$B^+$ toy saved to ROOT")
plt.show()


## CP toy: one TTree with a `charge` branch

For `generate_cp_toy`, the returned value remains `(plus_toy, minus_toy)`, while the ROOT representation stores both samples in one tree.


In [ ]:
cp_root = output_dir / "bkpipi_cp_toy.root"

plus_toy, minus_toy = generate_cp_toy(
    plus_model,
    minus_model,
    20_000,
    parameters=truth,
    seed=1902,
    output_root=cp_root,
    output_tree="DecayTree",
    charge_branch="charge",
)

print("B+ events:", plus_toy.size)
print("B- events:", minus_toy.size)
print("total:", plus_toy.size + minus_toy.size)


In [ ]:
with uproot.open(cp_root) as root_file:
    tree = root_file["DecayTree"]
    print("ROOT object type:", type(tree).__name__)
    print("entries:", tree.num_entries)
    print("branches:", tree.keys())

    charge = tree["charge"].array(library="np")
    print("charge values:", np.unique(charge))
    print("B+ in ROOT:", np.count_nonzero(charge == +1))
    print("B- in ROOT:", np.count_nonzero(charge == -1))


### Selecting $B^+$ and $B^-$ directly with uproot

The charge branch can be used as a normal ROOT selection.


In [ ]:
with uproot.open(cp_root) as root_file:
    tree = root_file["DecayTree"]

    plus_arrays = tree.arrays(
        ["s12", "s13", "s23", "charge"],
        cut="charge > 0",
        library="np",
    )
    minus_arrays = tree.arrays(
        ["s12", "s13", "s23", "charge"],
        cut="charge < 0",
        library="np",
    )

print("selected B+ events:", len(plus_arrays["s13"]))
print("selected B- events:", len(minus_arrays["s13"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
plot_dalitz(plus_toy, x="s13", y="s23", bins=80, ax=axes[0], title=r"$B^+$")
plot_dalitz(minus_toy, x="s13", y="s23", bins=80, ax=axes[1], title=r"$B^-$")
plt.show()


## Custom ROOT names and lighter files

The tree name, charge-branch name, weights and four-momenta are optional output choices. For example:


In [ ]:
_ = generate_cp_toy(
    plus_model,
    minus_model,
    2_000,
    parameters=truth,
    seed=1903,
    output_root=output_dir / "compact_cp_toy.root",
    output_tree="ToyTree",
    charge_branch="q",
    root_include_weights=False,
    root_include_momenta=False,
)
